In [15]:
from modules.EmbeddingsGenerator import EmbeddingsGenerator
embeddings_gene-rator = EmbeddingsGenerator(delete_models = False)

In [16]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [17]:
import pandas as pd
import time
from sklearn.preprocessing import minmax_scale
from numpy import linalg as LA


In [18]:
from modules.SimilarityEstimator import SimilarityEstimator
similarity_estimator = SimilarityEstimator()

In [19]:
data = pd.read_csv('../data/annotated_data.csv')
sample = data.sample(10)
sample

,id,sentence,sentence2,score
1589,1590,a person is ruffling the fur of a cat,a person is combing the fur of a cat,4.10
2839,2840,a man is climbing up a rope,a man is being roped before climbing,3.50
4217,4218,a woman is cracking eggs into a bowl,skewers are being placed onto a rack by a woman,2.50
2894,2895,a black sheep is standing near three white dogs,a black sheep is standing far from three white...,3.50
1055,1056,a lemur is biting a person 's finger,an animal is biting somebody 's hand,4.40
3468,3469,a dog is chasing another and is holding a stic...,a dog is chasing a stick and holding another d...,3.20
852,853,a dog is riding a skateboard,an animal is riding a skateboard,4.50
2341,2342,some boys are in front of dancing people,boys are dancing in front of some people,3.70
3178,3179,a boy wearing red shorts is jumping into a pad...,there is no boy wearing red shorts jumping int...,3.30
645,646,a man is sitting on the floor in a room and st...,a man is sitting on the floor and is playing a...,4.65


In [20]:
sentences_1 = sample.sentence
sentences_2 = sample.sentence2
truth = minmax_scale(sample.score)

In [21]:
def diff(pred, truth):
  return LA.norm(pred - truth)

In [22]:
cols = ['method', 'metric', 'score']
scores = pd.DataFrame(columns = cols)
model_names = embeddings_generator.get_supported_model_names()
distance_metrics = similarity_estimator.get_available_distance_metrics()
distance_metrics.remove('jensenshannon')

In [23]:
times = pd.DataFrame(columns = ['model', 'time'])

In [40]:
# for model_name in model_names:
model_name = model_names.pop()
start = time.process_time()
embeddings_1 = embeddings_generator.get_embeddings(model_name, sentences_1)
embeddings_2 = embeddings_generator.get_embeddings(model_name, sentences_2)
time_to_generate = time.process_time() - start
times = times.append(pd.DataFrame([[model_name, time_to_generate]], columns=['model', 'time']))

for distance_metric in distance_metrics:
  distances = similarity_estimator.calculate_pair_distances(embeddings_1, embeddings_2, distance_metric)

  score = diff(distances, truth)
  scores = scores.append(pd.DataFrame([[model_name, distance_metric, score]], columns = cols))

Embeddings Generator: Loading nnlm pretrained model
NNLM: Initializing
#################  <class 'tensorflow.python.framework.ops.EagerTensor'>
Embeddings Generator: nnlm already loaded
NNLM: Already Initialized
#################  <class 'tensorflow.python.framework.ops.EagerTensor'>


In [41]:
scores.to_csv('{}_scores.csv'.format(model_name))
times.to_csv('{}_times.csv'.format(model_name))

In [42]:
model_names

['use', 'word2vec']